<a href="https://githubtocolab.com/eskayML/issr_gsoc2026_communication_analysis/blob/main/01_Data_Resource_Selection.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Install dependencies if running in Colab
import sys
if 'google.colab' in sys.modules:
    !pip install librosa==0.11.0 noisereduce==3.0.3 soundfile==0.12.1 datasets==3.0.0 huggingface_hub

# GSoC 2026: ISSR Project 01 Test
## Notebook 1: Data Resource Selection and Sample Evaluation

**Applicant:** Samuel Kalu  
**Organization:** Institute for Social Science Research (ISSR)

> [!NOTE]
> This test is a continuation of my commitment to the ISSR lab's research goals. My **GSoC 2025** implementation ([read the final report here](https://eskayml.medium.com/gsoc-25-communication-analysis-tool-for-human-ai-interaction-driving-simulator-experiments-final-39c53d90672d)) laid the groundwork for large-scale simulator audio analysis. This 2026 iteration aims to refine the data ingestion and enhancement stages for even higher-density interactions.

### 1. Database Identification: The AMI Meeting Corpus
To address the project's requirement for a team communication environment, I have identified the **AMI Meeting Corpus** as the optimal data resource.

### 2. My Perspective: Why the `sdm` Subset for ISSR?

#### a. Simulating Acoustic Distance
The ISSR driving simulator involves **6 participants plus a narrator** in a single room. In our lab experiments, we often rely on distant microphones rather than individual headsets. To replicate this realistically, I have selected the **`sdm` (Single Distant Microphone)** subset of the AMI corpus.

The `sdm` data is intentionally "low-fidelity" compared to headset recordings; it contains natural room reverberation and lower signal-to-noise ratios. This is the exact type of "challenging" data my pipeline is designed to enhance.

#### b. Efficient Data Ingestion
Using the **Hugging Face `datasets` library** in streaming mode, we can surgically extract a specific meeting (e.g., **EN2008a**) without downloading the entire corpus. This demonstrates a modern, scalable approach to handling large-scale human-factors datasets.


In [ ]:
import os
import librosa
import numpy as np
import soundfile as sf
import matplotlib.pyplot as plt
import librosa.display
from datasets import load_dataset

def fetch_sdm_sample(target_path, meeting_id="EN2008a"):
    """
    Extracts a Single Distant Microphone (sdm) sample via HF streaming.
    """
    if os.path.exists(target_path):
        print(f"Sample already exists at {target_path}")
        return

    print(f"Streaming 'sdm' subset to extract meeting: {meeting_id}...")
    os.makedirs(os.path.dirname(target_path), exist_ok=True)
    
    # Load the 'sdm' (Single Distant Microphone) subset
    ds = load_dataset("edinburghcstr/ami", "sdm", split="train", streaming=True)
    
    # Filter for the specific meeting ID
    filtered_ds = ds.filter(lambda x: x["meeting_id"] == meeting_id)
    
    try:
        sample = next(iter(filtered_ds))
        audio_array = sample["audio"]["array"]
        sampling_rate = sample["audio"]["sampling_rate"]
        
        # Strictly follow PDF requirements (3-5 minute samples)
        # Slice to exactly 5 minutes (300 seconds)
        target_len = 300 * sampling_rate
        audio_array = audio_array[:target_len]
        
        # Save as our raw reference
        sf.write(target_path, audio_array, sampling_rate)
        print(f"Successfully extracted distant mic sample {meeting_id} to {target_path}")
    except StopIteration:
        print(f"Error: Meeting ID {meeting_id} not found in the 'sdm' stream.")

sample_path = "input/AMI_sample.wav"
fetch_sdm_sample(sample_path, meeting_id="EN2008a")

if os.path.exists(sample_path):
    y, sr = librosa.load(sample_path, sr=16000)
    print(f"\nSample Analysis:")
    print(f"Audio Duration: {librosa.get_duration(y=y, sr=sr):.2f} seconds")
    print(f"Sample Rate: {sr} Hz")


### 3. Exploratory Data Analysis (EDA)
Visualizing the distant microphone's spectral properties. Notice the higher noise floor and diffused formants typical of distant capture in a simulator environment.


In [ ]:
if os.path.exists(sample_path):
    plt.figure(figsize=(14, 8))

    # Plot Waveform
    plt.subplot(2, 1, 1)
    librosa.display.waveshow(y[:160000], sr=sr, color='darkorange')
    plt.title('Waveform: Distant Microphone (AMI EN2008a - SDM Subset)')
    plt.xlabel('Time (s)')
    plt.ylabel('Amplitude')

    # Plot Spectrogram
    plt.subplot(2, 1, 2)
    D = librosa.amplitude_to_db(np.abs(librosa.stft(y[:160000])), ref=np.max)
    librosa.display.specshow(D, sr=sr, x_axis='time', y_axis='hz')
    plt.colorbar(format='%+2.0f dB')
    plt.title('Spectrogram: High Noise Floor / Diffused Formants')

    plt.tight_layout()
    plt.show()
